# DAG assumption tests

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [3]:
df = pd.read_csv("dbs/data_p/commuter_model_features_r.csv")

In [4]:
df.columns

Index(['ID', 'entropy_mm', 'hill_q1', 'activity_nh_ratio',
       'activity_time_third', 'total_travel_time', 'xs_total_hws',
       'trip_chaining_presence', 'Gender', 'main_mode_r', 'Age', 'Education',
       'Household_type', 'weight_ind', 'access_h', 'mode', 'ak', 'ak_cat',
       'ak_log', 'codgeo', 'pt_sub', 'active_mode', 'h3_id', 'poverty_rate'],
      dtype='object')

## 1. Individual ⊥ Trips | CapabilitySet
This is not supported by the data.

A path from Individual to Trips is needed.

In [ ]:
# Model 1: Trips ~ CapabilitySet
m1 = smf.ols('xs_total_hws ~ ak_log', data=df).\
    fit(weight='weight_ind')

# Model 2: Trips ~ CapabilitySet + Individual
m2 = smf.ols('xs_total_hws ~ ak_log + C(Household_type)', data=df).fit(weight='weight_ind')    # non-significant effects are removed

# Compare nested models
anova_result = sm.stats.anova_lm(m1, m2)
print(anova_result)

   df_resid         ssr  df_diff   ss_diff        F    Pr(>F)
0    2442.0  429.624076      0.0       NaN      NaN       NaN
1    2435.0  424.949604      7.0  4.674472  3.82645  0.000383


In [5]:
# Model 1: Trips ~ CapabilitySet
m1 = smf.ols('total_travel_time ~ ak_log', data=df).\
    fit(weight='weight_ind')

# Model 2: Trips ~ CapabilitySet + Individual
m2 = smf.ols('total_travel_time ~ ak_log + C(Household_type)', data=df).fit(weight='weight_ind')    # non-significant effects are removed

# Compare nested models
anova_result = sm.stats.anova_lm(m1, m2)
print(anova_result)

   df_resid           ssr  df_diff       ss_diff         F    Pr(>F)
0    2442.0  3.863064e+06      0.0           NaN       NaN       NaN
1    2435.0  3.820077e+06      7.0  42986.958282  3.914403  0.000297


## 2. TransportMode ⊥ Participation | CapabilitySet, Individual

In [8]:

# Model 1: Participation ~ CapabilitySet + Individual
m1 = smf.ols('hill_q1 ~ ak_log + C(Gender) + C(Education) + poverty_rate + ' \
             'C(Household_type) + C(active_mode)', data=df).fit(weight='weight_ind')

# Model 2: + TransportMode (e.g., main_mode_r)
m2 = smf.ols('hill_q1 ~ ak_log + C(Gender) + C(Education) + poverty_rate + ' \
             'C(active_mode) + C(pt_sub) + C(Household_type) + C(main_mode_r)', data=df).fit(weight='weight_ind')

# Compare models
anova_result = sm.stats.anova_lm(m1, m2)
print(anova_result)

   df_resid            ssr  df_diff      ss_diff          F    Pr(>F)
0    2426.0  311765.920827      0.0          NaN        NaN       NaN
1    2424.0  308445.484191      2.0  3320.436636  13.047263  0.000002


## 3. TransportMode ⊥ Trips | CapabilitySet, Individual

In [9]:
# Model 1: Trips ~ CapabilitySet + Individual
m1 = smf.ols('xs_total_hws ~ ak_log + C(Gender) + C(Education) + poverty_rate + ' \
             'C(Household_type)', data=df).fit(weight='weight_ind')

# Model 2: Trips ~ CapabilitySet + Individual + TransportMode
m2 = smf.ols('xs_total_hws ~ ak_log + C(Gender) + C(Education) + poverty_rate + ' \
             'C(active_mode) + C(pt_sub) + C(Household_type) + C(main_mode_r)', data=df).fit(weight='weight_ind')

# Compare models
anova_result = sm.stats.anova_lm(m1, m2)
print(anova_result)

   df_resid         ssr  df_diff   ss_diff         F    Pr(>F)
0    2427.0  423.095603      0.0       NaN       NaN       NaN
1    2424.0  419.125062      3.0  3.970542  7.654511  0.000043


## 4. CapabilitySet ⊥ Individual | TransportMode

In [10]:
# Model 1: CapabilitySet ~ TransportMode
m1 = smf.ols('ak_log ~ C(main_mode_r) + C(pt_sub) + C(active_mode)', data=df).fit(weight='weight_ind')

# Model 2: CapabilitySet ~ TransportMode + Individual
m2 = smf.ols('ak_log ~ C(main_mode_r) + C(Gender) + C(Education) + poverty_rate + ' \
             'C(active_mode) + C(pt_sub) + C(Household_type) + C(active_mode)', data=df).fit(weight='weight_ind')

# Compare models
anova_result = sm.stats.anova_lm(m1, m2)
print(anova_result)

   df_resid           ssr  df_diff     ss_diff         F   Pr(>F)
0    2440.0  30933.413784      0.0         NaN       NaN      NaN
1    2425.0  30378.828065     15.0  554.585719  2.951333  0.00011
